<a href="https://colab.research.google.com/github/Kerey404/ragsys1/blob/main/rag_corporate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn pydantic openai qdrant-client sentence-transformers nest-asyncio

"Какая ставка по ипотеке на вторичное жилье?" или "Сколько дней длится отпуск?" "Что нужно для удаленного доступа к CRM?"

In [2]:

import nest_asyncio
import threading
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from openai import AsyncOpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
from google.colab import userdata

nest_asyncio.apply()

app = FastAPI(
    title="Bank - AI Assistant API",
    description="Внутренний микросервис для поиска ответов по регламентам банка (RAG + Qwen) для интеграции с CRM операторов.",
    version="1.0.0"
)

print("Загружаем модели и подключаемся к базе...")
qdrant_client = QdrantClient(":memory:")
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

groq_key = userdata.get('apigrok')

llm_client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_key
)
COLLECTION_NAME = "crm_docs"

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

test_texts = [
    "Максимальный срок кредита по программе Жас Отбасы составляет 25 лет.",
    "Для удаленного доступа к CRM операторам нужно использовать корпоративный VPN.",
    "Ставка по депозиту Баспана составляет 5.5% годовых."
]

points = []
for i, text in enumerate(test_texts, 1):
    vector = embedder.encode(text).tolist()
    points.append(PointStruct(id=i, vector=vector, payload={"text": text}))

qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)
print("Данные успешно векторизованы и загружены в Qdrant")

class QueryRequest(BaseModel):
    question: str

@app.post("/api/ask")
async def ask_crm_assistant(request: QueryRequest):
    try:
        query_vector = embedder.encode(request.question).tolist()


        search_result = qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            limit=2
        )

        context_texts = [hit.payload["text"] for hit in search_result.points]
        context_block = "\n".join([f"- {text}" for text in context_texts])

        system_prompt = (
            "Ты — умный корпоративный ИИ-ассистент. "
            "Твоя задача — отвечать на вопросы сотрудников строго на основе предоставленного контекста из базы данных компании. "
            "Если в контексте нет ответа на вопрос, честно скажи: 'Я не нашел этой информации в базе CRM'."
        )

        user_prompt = f"Контекст из базы:\n{context_block}\n\nВопрос сотрудника: {request.question}"

        response = await llm_client.chat.completions.create(
            model="llama-3.1-8b-instant", #тут можем поменять модель LLM
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1
        )

        return {
            "question": request.question,
            "answer": response.choices[0].message.content,
            "sources": context_texts
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server)
thread.start()
print("Сервер RAG успешно запущен в фоне на порту 8000! Готов принимать запросы.")

Загружаем модели и подключаемся к базе...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Данные успешно векторизованы и загружены в Qdrant
Сервер RAG успешно запущен в фоне на порту 8000! Готов принимать запросы.


In [3]:
import requests
import json

url = "http://localhost:8000/api/ask"

payload = {
    "question": "Какая ставка у депозита Баспана?"
}

print("Отправляем запрос в микросервис...")
response = requests.post(url, json=payload)

if response.status_code == 200:
    result = response.json()
    print("\n ОТВЕТ ОТ СЕРВЕРА С LLM:")
    print(f"🔹 Ответ: {result['answer']}")
    print(f"🔹 Откуда ИИ взял инфу: {result['sources']}")
else:
    print(f" Ошибка: {response.text}")

Отправляем запрос в микросервис...
INFO:     127.0.0.1:54214 - "POST /api/ask HTTP/1.1" 200 OK

 ОТВЕТ ОТ СЕРВЕРА С LLM:
🔹 Ответ: Ставка по депозиту Баспана составляет 5.5% годовых.
🔹 Откуда ИИ взял инфу: ['Ставка по депозиту Баспана составляет 5.5% годовых.', 'Максимальный срок кредита по программе Жас Отбасы составляет 25 лет.']


In [ ]:
import urllib

ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print(f"  (Endpoint IP): {ip}")
print(" Запускаем туннель в интернет...\n")

!npx localtunnel --port 8000

  (Endpoint IP): 35.190.163.195
 Запускаем туннель в интернет...

⠙⠹⠸⠼⠴your url is: https://crazy-places-jam.loca.lt
INFO:     85.159.27.200:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     85.159.27.200:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     85.159.27.200:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     85.159.27.200:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     85.159.27.200:0 - "POST /api/ask HTTP/1.1" 200 OK
INFO:     85.159.27.200:0 - "POST /api/ask HTTP/1.1" 200 OK
